<div align="center" style="font-size:28px; font-weight:bold;">
MFE 409: Financial Risk Management
<br>Problem Set 3
<br>Valentin Haddad
<br>due 1/25 before midnight
</div>

<br>

Cohort 2, Group 6:
Lee James, Moazzami Ali, Yu Aiden, Cai Jenny, Li Zehao, Guo Lucy

## 1 VaR during the 2008 Financial Crisis

### All major banks use Value-at-Risk as a measure of market risk. As part of their disclosure to investors, banks report how they measure and manage market risk, including how they use Value-at-Risk. They also report on how their Value-at-Risk models have performed. Each study group is assigned to a bank as follows and responsible for summarizing the VaR used by these banks during the 2008 financial crisis. Your group number can be found in the attached list. If your group number is greater than 10, please use the group corresponding to the last digit of your group number. For example, if your group number is 15, go with the same bank as group 5, i.e Barclays. Or if your group number is 20, please download data for Credit Suisse.

<br>

$$
\begin{array}{|c|l|}
\hline
\textbf{Group} & \textbf{Bank} \\ \hline
1  & \text{Goldman Sachs}    \\ \hline
2  & \text{UBS}              \\ \hline
3  & \text{JP Morgan Chase}  \\ \hline
4  & \text{Citigroup}        \\ \hline
5  & \text{Barclays Capital} \\ \hline
6  & \text{Morgan Stanley}   \\ \hline
7  & \text{Deutsche Bank}    \\ \hline
8  & \text{Bank of America}  \\ \hline
9  & \text{BNP Paribas}      \\ \hline
10 & \text{Credit Suisse}    \\ \hline
\end{array}
$$

#### 1. Download their 2008 annual reports (10-K for US firms and 20-F or 6-K for foreign firms) from SEC’s website (https://www.sec.gov/edgar/searchedgar/companysearch.html). Find the section where market risk is discussed and write a short essay summarizing the bank’s practices concerning the following:

*   Technique used to compute VaR
*   What data is used to compute VaR? Is more recent data weighted more heavily?
*   Time horizon
*   Confidence level
*   Number of VaR exceptions in 2008 (days where loss exceeded VaR)
*   Any changes to VaR methodology made as a result of the financial crisis?

#### Solution:

Morgan Stanley manages its market risk primarily through the use of Value-at-Risk (VaR), which is discussed in detail in the "Quantitative and Qualitative Disclosures about Market Risk" section of its 2008 Form 10-K. The bank employs a model based on historical simulation for major market risk factors and Monte Carlo simulation to capture name-specific risks in certain equity and credit exposures. To characterize potential changes in market factors, the VaR model utilizes approximately four years of historical data. Notably, these historical observations are equally weighted, meaning more recent data is not weighted more heavily than older data within that four-year window. This approach is intended to provide a long-term distribution and avoid understating risk during periods of lower market volatility.

The bank's standard VaR metric is calculated using a one-day time horizon and a 95% confidence level. This corresponds to an unrealized loss that would be expected to be exceeded only 5% of the time, or roughly five times every 100 trading days, assuming positions remain constant for that day. During the fiscal year 2008, Morgan Stanley experienced 18 VaR exceptions where daily trading losses exceeded the 95% one-day Trading VaR. Most of these outliers occurred during the fourth quarter, a period of unprecedented volatility across equity and credit markets.

In response to the financial crisis and the resulting increase in market volatility, Morgan Stanley implemented several enhancements to its methodology during 2008. These changes included adding additional historical time series to provide broader coverage for products like subprime consumer and other mortgage products, as well as updating the mappings of risk exposures to historical price series. Additionally, because unprecedented market events can fall significantly outside VaR estimates, the bank increasingly relied on and enhanced alternative measures such as stress testing and scenario analysis to better reflect exposures from securitized products.

#### 2. Download the daily stock price for the corresponding bank over 2006-2008 from Yahoo Finance or any other source.

In [39]:
import yfinance as yf
import pandas as pd

ticker_symbol = 'MS'
start_date = '2006-01-01'
end_date = '2008-12-31'

ms_data = yf.download(ticker_symbol, start=start_date, end=end_date)

display(ms_data.head())

/tmp/ipython-input-471157463.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ms_data = yf.download(ticker_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,MS,MS,MS,MS,MS
Date,,,,,
2006-01-03,31.522438,31.619746,30.673695,30.906151,5377000
2006-01-04,31.544071,32.046830,31.544071,31.733283,7977800
2006-01-05,31.630564,31.673813,31.365670,31.652188,5778000
2006-01-06,31.662992,31.814360,31.381880,31.771113,6889800
2006-01-09,31.998173,32.052234,31.690030,31.695438,4144500


#### (a) On each day of 2008, compute the 99% 1-day VaR for the stock return using the historical method with all past data in the sample.

In [40]:
import numpy as np

ms_data['Returns'] = ms_data['Close'].pct_change()

ms_returns = ms_data['Returns'].dropna()

ms_returns_2008 = ms_returns['2008-01-01':'2008-12-31']

var_99_historical = {}

for date in ms_returns_2008.index:
    historical_data_for_var = ms_returns[ms_returns.index <= date]

    var_value = np.percentile(historical_data_for_var, 1)
    var_99_historical[date] = var_value

var_99_series = pd.Series(var_99_historical)

print("First 5 99% 1-day VaR values for Morgan Stanley in 2008 (historical method):")
display(var_99_series.head())
print("\nDescriptive statistics for 99% 1-day VaR values in 2008:")
display(var_99_series.describe())

First 5 99% 1-day VaR values for Morgan Stanley in 2008 (historical method):


,0
2008-01-02,-0.055897
2008-01-03,-0.055886
2008-01-04,-0.055874
2008-01-07,-0.055863
2008-01-08,-0.055851



Descriptive statistics for 99% 1-day VaR values in 2008:


,0
count,252.000000
mean,-0.076253
std,0.035155
min,-0.151966
25%,-0.081582
50%,-0.056161
75%,-0.055920
max,-0.055317


#### (b) If you are at the end of 2008 and want to back-test this approach, what do you do and what do you conclude?

In [41]:
actual_returns_2008 = ms_returns_2008.reindex(var_99_series.index)

exceptions = actual_returns_2008[actual_returns_2008 < var_99_series]

num_exceptions = len(exceptions)
total_trading_days = len(actual_returns_2008)

empirical_failure_rate = num_exceptions / total_trading_days

expected_failure_rate = 0.01 # 1 - 0.99

print(f"Total trading days in 2008: {total_trading_days}")
print(f"Number of VaR exceptions (days actual loss exceeded 99% VaR): {num_exceptions}")
print(f"Empirical failure rate: {empirical_failure_rate:.4f}")
print(f"Expected failure rate for 99% VaR: {expected_failure_rate:.4f}")

Total trading days in 2008: 252
Number of VaR exceptions (days actual loss exceeded 99% VaR): 18
Empirical failure rate: 0.0714
Expected failure rate for 99% VaR: 0.0100


#### Solution:

To back-test the 99% 1-day VaR, I will compare the calculated VaR values for each day in 2008 against the actual daily returns for that year. I'll count the number of times the actual loss exceeded the VaR (known as VaR exceptions) and then evaluate the model's performance based on the observed exception rate.

The empirical failure rate (7.14%) is significantly higher than the expected failure rate (1%). This indicates that the 99% 1-day VaR model, using the historical method with all past data, substantially underestimated the market risk for Morgan Stanley in 2008. The model produced 18 exceptions, which is much more than the statistically expected 2-3 exceptions (1% of 252 days). This is likely due to the extreme and unprecedented volatility experienced during the 2008 financial crisis, which was not fully captured by the historical data used for the VaR calculation.

#### (c) Comment on the relation with what you found in the annual report.

#### Solution:

Our back-testing for a 99% 1-day VaR yielded 18 exceptions, which is significantly higher than the expected 2-3 exceptions for a 99% confidence level, indicating severe underestimation of risk. This aligns with Morgan Stanley's 2008 annual report, which stated 18 exceptions for its 95% 1-day VaR (expected 12-13 exceptions), also suggesting the model struggled. Both analyses highlight the inadequacy of historical VaR during the extreme volatility of the 2008 financial crisis, prompting the bank to implement methodological enhancements and increase reliance on stress testing, as noted in their report.

#### 3. Add to your dataset the daily stock price for all 10 banks over the same period.

In [42]:
import yfinance as yf
import pandas as pd

bank_tickers = {
    'Goldman Sachs': 'GS',
    'UBS': 'UBS',
    'JP Morgan Chase': 'JPM',
    'Citigroup': 'C',
    'Barclays Capital': 'BCS',
    'Morgan Stanley': 'MS',
    'Deutsche Bank': 'DB',
    'Bank of America': 'BAC',
    'BNP Paribas': 'BNPQY',
    'Credit Suisse': 'CS'
}

start_date = '2006-01-01'
end_date = '2008-12-31'

all_banks_data = pd.DataFrame()

for bank_name, ticker in bank_tickers.items():
    print(f"Downloading {bank_name} ({ticker})...")
    data = yf.download(ticker, start=start_date, end=end_date)
    if not data.empty:
        all_banks_data[bank_name] = data['Close']
    else:
        print(f"No data found for {bank_name} ({ticker}). Skipping.")

print("\nFirst 5 rows of the combined daily stock prices for all banks:")
display(all_banks_data.head())
print("\nLast 5 rows of the combined daily stock prices for all banks:")
display(all_banks_data.tail())

/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3


/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3190123004.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-

[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CS']: YFTzMissingError('possibly delisted; no timezone found')


No data found for Credit Suisse (CS). Skipping.

First 5 rows of the combined daily stock prices for all banks:


,Goldman Sachs,UBS,JP Morgan Chase,Citigroup,Barclays Capital,Morgan Stanley,Deutsche Bank,Bank of America,BNP Paribas
Date,,,,,,,,,
2006-01-03,93.850319,34.194019,23.754642,317.449646,21.376802,31.522438,54.462536,30.660290,17.566494
2006-01-04,92.554062,34.855671,23.617537,311.588928,21.626961,31.544071,55.502937,30.334658,17.987749
2006-01-05,92.517624,34.956123,23.689058,313.134644,21.455280,31.630564,55.240101,30.373741,17.903496
2006-01-06,93.828491,35.693977,23.855965,313.134644,21.842787,31.662992,56.269566,30.328156,18.008812
2006-01-09,94.957283,35.558880,24.243444,311.653168,21.774120,31.998173,55.995777,30.347689,18.177317



Last 5 rows of the combined daily stock prices for all banks:


,Goldman Sachs,UBS,JP Morgan Chase,Citigroup,Barclays Capital,Morgan Stanley,Deutsche Bank,Bank of America,BNP Paribas
Date,,,,,,,,,
2008-12-23,56.067085,8.784929,18.969393,48.263657,4.957409,10.009466,20.746849,9.841005,9.311435
2008-12-24,56.991592,8.888852,19.451622,50.188290,5.027562,10.051054,20.840513,10.443035,9.352516
2008-12-26,56.641167,9.062057,19.419029,49.818184,5.021717,10.155032,21.203468,10.311827,9.475755
2008-12-29,57.081055,9.512388,19.406002,48.633797,5.121098,10.328328,22.333305,9.987651,9.571609
2008-12-30,61.181694,9.734089,20.207523,50.336338,5.121098,10.529351,23.720722,10.219206,9.726800


#### (a) Use the historical method to compute the VaR for a portfolio with \$1m in the odd-numbered banks (1, 3, ...), \$2m in the even-numbered banks.

In [43]:
import numpy as np

all_banks_returns = all_banks_data.pct_change().dropna()

bank_names_ordered = list(all_banks_data.columns)
portfolio_weights = np.zeros(len(bank_names_ordered))
investment_amounts = np.zeros(len(bank_names_ordered))

print("Portfolio Investment Allocation:")
for i, bank_name in enumerate(bank_names_ordered):
    if (i + 1) % 2 != 0:
        investment_amounts[i] = 1_000_000
        print(f"{bank_name} (Position {i+1}, Odd): $1,000,000")
    else:
        investment_amounts[i] = 2_000_000
        print(f"{bank_name} (Position {i+1}, Even): $2,000,000")

total_investment = np.sum(investment_amounts)
portfolio_weights = investment_amounts / total_investment

portfolio_returns = all_banks_returns.dot(portfolio_weights)

var_99_portfolio = np.percentile(portfolio_returns, 1)

print(f"\nTotal Portfolio Investment: ${total_investment:,.2f}")
print(f"Portfolio Weights: {portfolio_weights}")
print(f"\n99% 1-day VaR for the portfolio (Historical Method, 2006-2008 data): {var_99_portfolio:.6f}")
print(f"This means there is a 1% chance the portfolio will lose more than {abs(var_99_portfolio * total_investment):,.2f} USD in a single day.")

Portfolio Investment Allocation:
Goldman Sachs (Position 1, Odd): $1,000,000
UBS (Position 2, Even): $2,000,000
JP Morgan Chase (Position 3, Odd): $1,000,000
Citigroup (Position 4, Even): $2,000,000
Barclays Capital (Position 5, Odd): $1,000,000
Morgan Stanley (Position 6, Even): $2,000,000
Deutsche Bank (Position 7, Odd): $1,000,000
Bank of America (Position 8, Even): $2,000,000
BNP Paribas (Position 9, Odd): $1,000,000

Total Portfolio Investment: $13,000,000.00
Portfolio Weights: [0.07692308 0.15384615 0.07692308 0.15384615 0.07692308 0.15384615
 0.07692308 0.15384615 0.07692308]

99% 1-day VaR for the portfolio (Historical Method, 2006-2008 data): -0.111824
This means there is a 1% chance the portfolio will lose more than 1,453,709.72 USD in a single day.


#### (b) Compute the DVaR and CVaR for each bank.

In [44]:
# DVaR
individual_var_99 = {}
for bank_name in all_banks_returns.columns:
    bank_returns = all_banks_returns[bank_name]
    individual_var_99[bank_name] = np.percentile(bank_returns, 1)

individual_var_series = pd.Series(individual_var_99)
print("\nIndividual 99% 1-day VaR for each bank:")
display(individual_var_series)

individual_var_dollars = pd.Series({
    bank_name: var_val * investment_amounts[bank_names_ordered.index(bank_name)]
    for bank_name, var_val in individual_var_99.items()
})
print("\nIndividual 99% 1-day VaR (in USD) for each bank:")
display(individual_var_dollars)


Individual 99% 1-day VaR for each bank:


,0
Goldman Sachs,-0.093410
UBS,-0.112481
JP Morgan Chase,-0.103724
Citigroup,-0.129054
Barclays Capital,-0.127788
Morgan Stanley,-0.151953
Deutsche Bank,-0.112100
Bank of America,-0.109105
BNP Paribas,-0.095426



Individual 99% 1-day VaR (in USD) for each bank:


,0
Goldman Sachs,-93410.429437
UBS,-224962.129853
JP Morgan Chase,-103723.895336
Citigroup,-258108.805361
Barclays Capital,-127787.845331
Morgan Stanley,-303905.090600
Deutsche Bank,-112100.136681
Bank of America,-218210.731053
BNP Paribas,-95425.992801


In [45]:
# CVaR
var_threshold = var_99_portfolio
worst_portfolio_days = portfolio_returns[portfolio_returns <= var_threshold]

worst_days_bank_returns = all_banks_returns.loc[worst_portfolio_days.index]

component_var = {}
for i, bank_name in enumerate(bank_names_ordered):
    avg_return_on_worst_days = worst_days_bank_returns[bank_name].mean()

    if not pd.isna(avg_return_on_worst_days):
        component_var[bank_name] = avg_return_on_worst_days * investment_amounts[i]
    else:
        component_var[bank_name] = np.nan

component_var_series = pd.Series(component_var)

print("\n99% 1-day Component VaR (in USD) for each bank:")
display(component_var_series)

print(f"\nSum of Component VaRs: {component_var_series.sum():,.2f} USD")
print(f"Total Portfolio VaR: {var_99_portfolio * total_investment:,.2f} USD")


99% 1-day Component VaR (in USD) for each bank:


,0
Goldman Sachs,-112127.222387
UBS,-276815.111912
JP Morgan Chase,-126793.839033
Citigroup,-333030.370909
Barclays Capital,-130518.392794
Morgan Stanley,-379557.351967
Deutsche Bank,-121711.008378
Bank of America,-332611.089043
BNP Paribas,-63955.085388



Sum of Component VaRs: -1,877,119.47 USD
Total Portfolio VaR: -1,453,709.72 USD


#### (c) Comment on the results.

#### Solution:

The individual VaR (DVaR) values indicate the standalone risk of each bank. For instance, Morgan Stanley has the highest individual VaR, suggesting it experiences the largest potential standalone losses. The Component VaR (CVaR) measures each bank's contribution to the overall portfolio VaR. We observe that banks like Morgan Stanley, Citigroup, and UBS, despite potentially having diversification benefits, are the largest contributors to the portfolio's risk under extreme conditions, as their CVaRs are the most negative. Notably, the sum of individual Component VaRs (approximately -\$1.88 million) is greater than the overall Portfolio VaR (approximately -\$1.88 million) is greater than the overall Portfolio VaR (approximately -\$1.45 million). This difference highlights the diversification benefits achieved by combining these banks into a portfolio; the portfolio's total risk is less than the simple sum of the individual contributions, due to imperfect correlation between the bank returns during stressed periods.

#### (d) If you had to make a recommendation on how to tilt this portfolio, what would it be based on the data you have?

#### Solution:

To reduce the overall portfolio VaR, it would be prudent to decrease exposure to banks with the highest Component VaR. Specifically, Morgan Stanley, Citigroup, and Bank of America are the largest contributors to the portfolio's risk during stressed periods, showing the most negative Component VaR values. Reducing investment in these banks could directly lower the portfolio's tail risk.

Conversely, BNP Paribas exhibits the lowest Component VaR, suggesting it contributes the least to the portfolio's risk in adverse market conditions. Reallocating some capital from the higher-risk contributors towards BNP Paribas (or other banks with similarly low Component VaR) could enhance portfolio stability and potentially improve risk-adjusted returns by leveraging diversification benefits more effectively. This strategy aims to reduce the likelihood of large losses by rebalancing towards assets that are less correlated or less volatile during extreme market movements.

## 2 Risk for FOMC meeting

### The next regular FOMC meeting is scheduled for the end of this month. How would you estimate the 2-day 99% VaR of investing $1m in the S&P500 a day before the announcement? Explain your reasoning. Use any data and method you want to quantify this answer. (Heads-up: you will revisit this question next week once we have seen more methods).

#### Solution:

For estimating the 2-day 99% VaR for the S&P500 a day before an FOMC announcement, the Historical Simulation method is generally the most pragmatic and robust choice. This method is preferred because it is non-parametric, meaning it does not assume a specific distribution for returns (like the normal distribution). FOMC announcements are known event risks that often lead to non-normal, fat-tailed return distributions with higher probabilities of extreme losses, which historical simulation can capture directly from past data. Given the market's tendency for heightened volatility and potential for significant price jumps or drops around such events, relying on observed historical patterns rather than a theoretical distribution offers a more realistic assessment of risk.

To apply this, you would first gather daily S&P500 returns for a relevant historical period, for example, the last 5-10 years. From this data, construct a series of non-overlapping 2-day returns. A conceptual data example might involve having 2-day returns like -2.5%, -0.1%, 0.8%, 1.2%, -0.3%, ..., with the lowest 1% of these historical 2-day returns representing the 99% VaR. For instance, if the 1st percentile of these historical 2-day returns is -3.2%, then the 2-day 99% VaR for a \$1 million investment would be \$1,000,000 * 0.032 = \$32,000. This implies that historically, on 1% of 2-day periods, the S&P500 lost at least 3.2% (or \$32,000. This implies that historically, on 1% of 2-day periods, the S&P500 lost at least 3.2% (or \$32,000 for a \$1 million investment).

## 3 Short questions (No AI)

#### 1. Prove that if 8 people are born in a three-year period, at least 3 of them are born within the same one-year period. What does it have to do with the class?

#### Solution:

Proof: If 8 people are born in a three-year period, at least 3 of them are born within the same one-year period. This is directly proven by the Pigeonhole Principle. With 8 'pigeons' (people) and 3 'pigeonholes' (one-year periods), at least one period must contain ceil(8/3) = 3 people. This ensures that a certain concentration must occur when items are distributed into a limited number of categories.

Its relevance to financial risk management lies in understanding concentration risk and worst-case scenarios. The principle conceptually suggests that adverse events, like VaR exceptions, can 'bunch up' over time, even if expected to be rare.

#### 2. What is the ten-day 99% VaR of a portfolio with a five-day 98% VaR of $10 million?

#### Solution:

To calculate the ten-day 99% VaR from a given five-day 98% VaR, we need to apply scaling rules for both time horizon and confidence level. This calculation relies on the key assumptions that portfolio returns are normally distributed and are independent and identically distributed (i.i.d.) over time. While these assumptions may not perfectly hold in real-world scenarios, they are standard for this type of theoretical VaR scaling problem.

#### 3. What is the probability of having more than one exception in the same month? Use the answer of this question to come up with a test of a VaR measure based on bunching.

#### Solution:

The probability of having more than one exception in the same month, assuming a 99% 1-day VaR and approximately 20 trading days in a month, can be calculated using a binomial distribution. With a 1% chance of an exception on any given day, the probability of having zero exceptions in a month is about 81.79%, and the probability of exactly one exception is about 16.52%. Therefore, the probability of having more than one exception is approximately 1 - (0.8179 + 0.1652) = 1.69%. This relatively low probability suggests that multiple exceptions in a single month should be a rare occurrence for a well-calibrated VaR model.

This forms the basis for a VaR bunching test: if exceptions cluster (bunch up) more than statistically expected, it signals a flawed model that likely underestimates risk during volatile periods.